# SVD — The X-Ray Vision for Matrices

Every matrix hides a secret structure. SVD reveals it.

Think of any matrix as a transformation — it stretches, rotates, and flips space. SVD breaks that transformation into exactly **three simple steps**:

1. **Rotate** (Vᵀ) — align with the matrix's "natural axes"
2. **Stretch** (Σ) — scale along each axis (these are the singular values)
3. **Rotate again** (U) — orient into the output space

**A = U · Σ · Vᵀ**

The magical part? The singular values (Σ) tell you **how important** each direction is. The biggest singular value captures the most "energy" of the matrix. This is why you can throw away the small ones and still keep a great approximation — that's how image compression, Netflix recommendations, and noise removal work.

## How we build it from scratch

No black-box `np.linalg.svd` here. We use two simple ideas:

- **Power Iteration**: Multiply a random vector by a matrix over and over — it naturally drifts toward the most important direction. Like a compass finding north.
- **Deflation**: Once you've found the most important direction, subtract it out and repeat. Peel the matrix like an onion, layer by layer.

In [236]:
import numpy as np
import math

In [237]:
A = np.array([[2, 1], [1, 2]])
A

array([[2, 1],
       [1, 2]])

In [238]:
def power_iteration(M, num_iters, v):
    if num_iters == 0:
        return v

    v_new = np.dot(M, v)
    v_new = v_new / np.linalg.norm(v_new)

    if np.allclose(v, v_new):
        return v_new
    
    return power_iteration(M, num_iters - 1, v_new)

In [239]:
e_v1 = power_iteration(A, 1000, np.random.random(2))
e_v1

array([0.70710824, 0.70710532])

In [240]:
e_val1 = np.dot(np.dot(np.transpose(e_v1), A), e_v1)
e_val1

np.float64(2.999999999991432)

In [241]:
A_deflated = A - e_val1 * np.outer(e_v1, np.transpose(e_v1))
A_deflated

array([[ 0.49999379, -0.5       ],
       [-0.5       ,  0.50000621]])

In [242]:
e_v2 = power_iteration(A_deflated, 10, np.random.random(2))
e_v2

array([-0.70710239,  0.70711117])

In [243]:
e_val2 = np.dot(np.dot(np.transpose(e_v2), A_deflated), e_v2)
e_val2

np.float64(1.0000000000257012)

In [244]:
A_deflated_2 = A - e_val2 * np.outer(e_v2, np.transpose(e_v2))
A_deflated_2

array([[1.50000621, 1.5       ],
       [1.5       , 1.49999379]])

In [245]:
AtA = np.array(A).T @ np.array(A)
AtA

array([[5, 4],
       [4, 5]])

In [246]:
Ata_e_v1 = power_iteration(AtA, 10, np.random.random(2))
Ata_e_val1 = np.dot(np.dot(np.transpose(Ata_e_v1), AtA), Ata_e_v1)
Ata_e_v1, Ata_e_val1


(array([0.70710663, 0.70710693]), np.float64(8.99999999999963))

In [247]:
Ata_deflated = AtA - Ata_e_val1 * np.outer(Ata_e_v1, np.transpose(Ata_e_v1))
Ata_deflated

array([[ 0.50000193, -0.5       ],
       [-0.5       ,  0.49999807]])

In [248]:
Ata_e_v2 = power_iteration(Ata_deflated, 10, np.random.random(2))
Ata_e_val2 = np.dot(np.dot(np.transpose(Ata_e_v2), Ata_deflated), Ata_e_v2)
Ata_e_v2, Ata_e_val2

(array([-0.70710815,  0.70710542]), np.float64(1.0000000000033158))

In [249]:
V = np.column_stack([Ata_e_v1, Ata_e_v2])
V

array([[ 0.70710663, -0.70710815],
       [ 0.70710693,  0.70710542]])

In [250]:
sigma1 = math.sqrt(Ata_e_val1)
sigma2 = math.sqrt(Ata_e_val2)
sigma1, sigma2

(2.9999999999999383, 1.0000000000016578)

In [251]:
u1 = A @ V[:, 0] / sigma1
u2 = A @ V[:, 1] / sigma2
u1, u2

(array([0.70710673, 0.70710683]), array([-0.70711088,  0.70710268]))

In [252]:
U = np.column_stack([u1, u2])
U

array([[ 0.70710673, -0.70711088],
       [ 0.70710683,  0.70710268]])

In [253]:
sigmas = [sigma1, sigma2]

U @ np.diag(sigmas) @ np.transpose(V)

array([[2.00000343, 0.99999828],
       [1.00000172, 1.99999657]])